In [1]:
import os
os.chdir('/workspace/7dea3ab1-d043-4137-8092-d06402dd09c6')
print(os.listdir('.'))


['.prompts', 'ldh_zeros_N5000_dps50.npy', '-PROMPT-v6-DATASET.md', 'ldh_off_line_zeros.csv', '.config', 'lchi5_zeros_N5000_dps80.npy', 'memory', '.kernel_llm_logs_1.txt', 'zeta_delta_strong_N5000_dps50.npy', 'zeta_zeros_N5000_dps50.npy']


In [2]:
import numpy as np
# Check available files
for f in ['zeta_zeros_N5000_dps50.npy', 'ldh_zeros_N5000_dps50.npy', 'lchi5_zeros_N5000_dps80.npy', 'zeta_delta_strong_N5000_dps50.npy']:
 a = np.load(f)
 print(f, a.shape, a.dtype, a[0], '...', a[-1])


zeta_zeros_N5000_dps50.npy (5000,) <U51 14.134725141734693790457251983562470270784257115699 ... 5447.8619983012998564121586734642921676829672006076
ldh_zeros_N5000_dps50.npy (5000,) <U51 5.0941598445710949256987955170797974750670744531091 ... 4981.1674898625791053105083724292317970900546075558
lchi5_zeros_N5000_dps80.npy (5000,) <U81 6.6484533447277147161232784599793178472985854232444983723747129467 ... 4388.2090125941801377592509179068522248538541759189201573787539145
zeta_delta_strong_N5000_dps50.npy (5000,) [('re', '<U10'), ('im', '<U51')] ('0.5', '14.134725141734693790457251983562470270784257115699') ... ('0.5', '5447.8619983012998564121586734642921676829672006076')


In [3]:
# L(Delta, s) zeros - need to check what's available
import glob
files = glob.glob('*.npy') + glob.glob('*ldelta*') + glob.glob('*delta*')
print(files)
import os
# Also check downloads
for d in ['downloads', 'data', '.']:
 if os.path.exists(d):
 for f in os.listdir(d):
 if 'delta' in f.lower() or 'ldelta' in f.lower():
 print(d, f)


['ldh_zeros_N5000_dps50.npy', 'lchi5_zeros_N5000_dps80.npy', 'zeta_delta_strong_N5000_dps50.npy', 'zeta_zeros_N5000_dps50.npy', 'zeta_delta_strong_N5000_dps50.npy']
. zeta_delta_strong_N5000_dps50.npy


In [4]:
# Check for any related artifacts in /workspace
import subprocess
out = subprocess.run(['find', '/workspace', '-name', '*ldelta*', '-o', '-name', '*delta*zero*'], capture_output=True, text=True)
print("STDOUT:", out.stdout)
print("STDERR:", out.stderr[:500])


STDOUT: 
STDERR: find: ‘/workspace/lost+found’: Permission denied



In [5]:
# Check if cypari2 is available for generating L(Delta) zeros
try:
 import cypari2
 pari = cypari2.Pari()
 print("cypari2 available")
 print(pari.version())
except Exception as e:
 print("Error:", e)


Error: No module named 'cypari2'


In [6]:
# Try installing cypari2 
import subprocess
r = subprocess.run(['pip', 'install', '-q', 'cypari2', 'cysignals'], capture_output=True, text=True, timeout=300)
print("RC:", r.returncode)
print("STDOUT:", r.stdout[-2000:])
print("STDERR:", r.stderr[-2000:])


RC: 0
STDOUT: 
STDERR: 


In [7]:
import cypari2
pari = cypari2.Pari()
print(pari.version())
# Try LMFDB modular form 1.12.a.a (Ramanujan Delta)
L = pari('lfunmf(mfinit([1,12,1],0)[3][1])') # might not work
print(L)


(2, 17, 2)


PariError: lfunmf: incorrect type in checkMF [please use mfinit] (t_VEC)

In [8]:
# mfinit for weight 12 level 1
# In PARI: mf = mfinit([1,12],1); then lf = lfunmf(mf); the Delta cusp form
pari('mf = mfinit([1,12],1)')
print(pari('mf'))


[[1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], 1], [], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1]]], [Vecsmall([1, 1, 1])], [Vecsmall([2]), [Mat(1), 1, 1, 0], [0; 1]], [0, 0, 0, 0, 0]]


In [9]:
# Get Delta as a newform - mfeigenbasis
pari('forms = mfeigenbasis(mf)')
print(pari('#forms'))
pari('Delta = forms[1]')
# Now create L function
pari('LDelta = lfunmf(Delta)')
print(pari('LDelta'))


1


PariError: lfunmf: incorrect type in checkMF [please use mfinit] (t_VEC)

In [10]:
# lfunmf needs both mf and the form. Try: lfunmf([mf, Delta]) or just lfunmf(F)
# In PARI doc: lfunmf(F) where F is a modular form...
# Actually need to use lfunmf(mf, [vector of forms])
try:
 r = pari('lfunmf(mf, Delta)')
 print(r)
except Exception as e:
 print("e1", e)


[[Vecsmall([14]), [[[Vecsmall([15]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[Vecsmall([9]), [1, 12, [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1], y]], [[[1, [0]], [1, [], []], [[]~, Vecsmall([])], [[], [], [;], [], [], []], [;]], []~, 1, t - 1]]], [1]~, 1], [], 1]], 0, [0, 1], 12, 1, 1]


In [11]:
pari('LDelta = lfunmf(mf, Delta)')
# Try a single zero to test - need to understand normalization
# In PARI, lfunmf for weight-k form L(s) typically uses motivic normalization
# Conductor=1, weight=12 form Delta: critical line is Re(s)=6 in arithmetic normalization, Re(s)=1/2 in analytic
# lfunzeros returns ordinates t such that L(1/2+it)=0 in analytic normalization
import time
t0 = time.time()
z = pari('lfunzeros(LDelta, 30)')
print("Time:", time.time()-t0)
print(z)


Time: 0.012963533401489258
[9.22237939992110, 13.9075498613921, 17.4427769782345, 19.6565131419550, 22.3361036372099, 25.2746365481124, 26.8043911583504, 28.8316826241869]


In [12]:
# Only got 8 zeros up to t=30. Per LMFDB, first zero of L(Delta,s) (analytic normalization) is t~9.222
# This matches LMFDB 1.12.a.a. Great.
# Strategy: use lfunzeros(LDelta, [t0, t1]) to get zeros in a range, then merge.
# Test getting first 100
t0=time.time()
z = pari('lfunzeros(LDelta, 100)')
print("count:", pari('#z'), "time:", time.time()-t0)
print("last:", z[-1])


count: 2 time: 0.09440755844116211


IndexError: index out of range

In [13]:
# #z gave 2 — probably the dim of the array description; let's use length()
import time
t0 = time.time()
z = pari('lfunzeros(LDelta, 100)')
print("time:", time.time()-t0)
# Convert via PARI
n = int(pari('#lfunzeros(LDelta, 100)'))
print("n=", n)
# Hmm that's weird. Let me extract differently
zlist = pari('Vec(lfunzeros(LDelta, 100))')
print("len:", len(zlist))
print("first:", zlist[0], "last:", zlist[-1])


time: 0.09465909004211426
n= 59
len: 59


IndexError: index out of range

In [14]:
# OK so 59 zeros up to t=100. Use Strprintf for full precision extraction.
# Time scaling: T^2.85. For T=4400 (where ~5000 zeros lie), this would be huge.
# But for Front III we just need imaginary parts. We can use lower precision than dps=80
# since we only need unfolded spacings, which are O(1). Float64 may even be sufficient.
# Let me check: at t=4400, how dense are zeros? density ~ k*log(t)/(2*pi) for weight-12 form
# N(T) ~ T*log(T)/(pi) for weight-12 L-function? Actually for degree-2 L-function: N(T) ~ T*log(T)/(2*pi) + ...
# Let's estimate: for Delta, the density is roughly log(T)*T/pi. For T=4400, N~4400*log(4400)/pi ~ 4400*8.4/3.14 ~ 11770. Hmm.
# Wait: for L(Delta), counting function N(T) ~ (T/pi)*log(T*sqrt(N)/(2*pi*e)) * d where d=degree=2.
# For deg=2, level 1, weight 12: N(T) ~ (T/pi)*log(T/(2*pi*e)) + lower order... ~ T*log(T)/pi
# Set T*log(T)/pi = 5000 -> T ~ 1800 perhaps. Let's just check empirically.

# Get zeros up to t=200 first
import time
t0 = time.time()
n = int(pari('#lfunzeros(LDelta, 200)'))
print("zeros up to T=200:", n, "time:", time.time()-t0)


zeros up to T=200: 159 time: 0.5000202655792236


In [15]:
# T=200 -> 159 zeros. T*log(T)/pi at T=200 = 200*5.3/3.14 ~ 337. Hmm but we got 159.
# Let me check T=1000
import time
t0=time.time()
n = int(pari('#lfunzeros(LDelta, 1000)'))
print("T=1000:", n, "time:", time.time()-t0)


T=1000: 1298 time: 90.06171441078186


In [16]:
# 1298 zeros at T=1000 in 90 sec - matches the "first 1298 zeros at dps=80" note!
# Scaling T^2.85: T=2000 -> ~90 * 2^2.85 = ~648s, T=2500 ->~1265s, T=3000->~2128s
# We need ~5000 zeros which probably means T~2500-3000.
# Let me estimate: N(T) for L(Delta) scales as T*log(T)/pi roughly (degree 2, conductor 1, no characters)
# 1000 -> 1298, scaling N(T) ~ T log T... so for N=5000, T*log T ~ (5000/1298)*1000*log(1000) ~ 26600, so T ~ 2900

# But we have time budget. Let's first see if the LMFDB analytic normalization matches what we need
# Save zeros up to T=1000 as a starting point
z = pari('lfunzeros(LDelta, 1000)')
# Extract precise values
import time
t0 = time.time()
# Use Strprintf with default precision (38 digits typical for double-precision PARI)
# Currently realprecision is default (38)
prec_str = pari('default(realprecision)')
print("realprecision:", prec_str)


realprecision: 15


In [17]:
# Default precision was only 15. Let's bump it. Need at least dps=38 (~ realprec 38) for the search to match prior protocol.
# Actually for Front III analysis (unfolded spacings), float64 is more than enough.
# But let's use higher precision for safety. Let's use realprecision=50.

# First extract current zeros (the 1298 already computed)
import numpy as np, time
t0 = time.time()
# Extract as strings using Strprintf to get many digits
# We re-execute lfunzeros at higher precision
pari('default(realprecision, 50)')
print("recomputing with prec=50")
t0=time.time()
pari('z1000 = lfunzeros(LDelta, 1000)')
print("time:", time.time()-t0, "count:", int(pari('#z1000')))


recomputing with prec=50
time: 114.94271755218506 count: 1298


In [18]:
# Extract them to Python strings
import time
t0 = time.time()
N = 1298
zeros_strs = []
for i in range(1, N+1):
 s = str(pari(f'Strprintf("%.50g", z1000[{i}])'))
 zeros_strs.append(s)
print("extract time:", time.time()-t0)
print("first 3:", zeros_strs[:3])
print("last:", zeros_strs[-1])


extract time: 0.005284547805786133
first 3: ['9.2223793999211025222437671927434781355287706224320', '13.907549861392134406446681328770219491757552353514', '17.442776978234473313551525137127262718708866524275']
last: 999.41630482411118039241011745672400950736853054932


In [19]:
# Now extend to get N=5000. We need to go up to T ~ 2900-3000.
# Strategy: search range-by-range using lfunzeros(L, [t_start, t_end]).
# Cost scales as T^2.85. Plan: search in chunks of width ~200-300.
# Total time for T=3000 (cumulative): T^2.85 * const. ~ (3/1)^2.85 = 24.5x slower per unit length... but cumulative
# Actually if total time up to T scales as T^~3.85, then T=2900: 90 * (2900/1000)^3.85 = 90 * 65 = ~5800s. Too slow.
# 
# But for Front III, we don't need exhaustive zeros. We just need a long sample of consecutive zeros.
# We can also use a lower precision search (dps=15 might find them and be much faster).
# Let me test: at dps=15, search T=1000-2000
pari('default(realprecision, 38)')
import time
t0 = time.time()
pari('z_a = lfunzeros(LDelta, [1000, 1500])')
print("time:", time.time()-t0, "count:", int(pari('#z_a')))


 *** lfunzeros: Warning: 

PariError: lfunzeros: the PARI stack overflows (current size: 8003584; maximum size: 8003584)
You can use pari.allocatemem() to change the stack size and try again

increasing stack size to 8003584.


In [20]:
# Need more stack
pari.allocatemem(2*1024*1024*1024) # 2GB
import time
t0 = time.time()
pari('z_a = lfunzeros(LDelta, [1000, 1500])')
print("time:", time.time()-t0, "count:", int(pari('#z_a')))


PARI stack size set to 2147483648 bytes, maximum size set to 2147483648
time: 351.66126775741577 count: 841


In [21]:
# 351s for 500 width starting at T=1000 -> we got 1298+841=2139 zeros at T=1500
# Cost grows as T^2.85 so next: 1500-2000 will cost ~ 351 * (1750/1250)^2.85 ~ 351*2.6 = 913s
# Then 2000-2500: 351 * (2250/1250)^2.85 ~ 351*5 = 1750s
# Cumulative: 90 + 351 + 913 + 1750 = 3104s. With dps=38 (current). Already at our 3600s budget.
# 
# We only have ~3000s left. Better strategy: do remaining with smaller chunks and accept N<5000 if needed.
# Actually wait — at 1500 we have 2139 zeros. We need 5000. Let me re-estimate where N=5000 lies.
# 1000 -> 1298. 1500 -> 2139. Increment over 500: 841 zeros, density ~1.68/unit at T=1250
# Asymptotic N(T)/T ~ log(T)/pi - log(2*pi*e)/pi roughly. Constant ~ growing slowly.
# Extrapolation: T=2500 maybe ~3800, T=3000 ~ 4600, T=3500 ~ 5400.
# 
# Given time budget, I will: extract z_a (1000-1500) zeros, then chunk 1500-2500 in 250-step chunks
# Then assess.

# Extract z_a first
N_a = int(pari('#z_a'))
zeros_a = []
for i in range(1, N_a+1):
 zeros_a.append(str(pari(f'Strprintf("%.50g", z_a[{i}])')))
print("extracted", len(zeros_a), "first:", zeros_a[0], "last:", zeros_a[-1])


extracted 841 first: 1000.16390643579543089723183527557876986 last: 1499.34291680557581436028637405848330675


In [22]:
all_zeros = zeros_strs + zeros_a
print("total:", len(all_zeros))

# Save intermediate
import numpy as np
np.save('ldelta_zeros_partial_T1500.npy', np.array(all_zeros, dtype='<U51'))
print("saved")

# Now continue with 1500-2000
import time
t0 = time.time()
pari('z_b = lfunzeros(LDelta, [1500, 2000])')
print("time:", time.time()-t0, "count:", int(pari('#z_b')))


TimeoutError: Code execution timed out after 900.0 seconds

In [23]:
# Timeout. The cell timed out at 900s. Let me check state.
import time
# Likely lfunzeros is still running or done. Try check
try:
 n = int(pari('#z_b'))
 print("z_b count:", n)
except Exception as e:
 print("err:", e)


z_b count: 896


In [24]:
# Got 896 zeros from 1500-2000. Extract them.
N_b = int(pari('#z_b'))
zeros_b = []
for i in range(1, N_b+1):
 zeros_b.append(str(pari(f'Strprintf("%.50g", z_b[{i}])')))
all_zeros = all_zeros + zeros_b
print("total:", len(all_zeros), "last:", all_zeros[-1])
import numpy as np
np.save('ldelta_zeros_partial_T2000.npy', np.array(all_zeros, dtype='<U51'))


total: 3035 last: 1999.69332694937638647239814949495766571


In [25]:
# 3035 zeros at T=2000. Need ~5000. Need to go to about T~2800-3000.
# Time consumed so far: ~90 (T=1000) + 351 (T=1500) + ~900 (T=2000) = ~1340s
# Remaining time roughly: 3600 - 1340 - overhead ~= 2200s.
# 
# Next chunk 2000-2500: scales as (2250/1750)^2.85 from previous chunk. Previous chunk took ~900s.
# (2250/1750)^2.85 = (1.286)^2.85 = ~2.05. So next ~1845s. That's too much.
# 
# Compromise: use a smaller chunk to be safe. Let's try 2000-2300 first.
import time
t0 = time.time()
try:
 pari('z_c = lfunzeros(LDelta, [2000, 2300])')
 print("ok time:", time.time()-t0, "count:", int(pari('#z_c')))
except Exception as e:
 print("err after", time.time()-t0, ":", e)


ok time: 880.4757828712463 count: 557


In [26]:
# 3035 + 557 = 3592 zeros at T=2300. Still short of 5000.
# Time is tight. Extract these and proceed with N=3592 for L(Delta).
# This is a clear, documented limitation.
N_c = int(pari('#z_c'))
zeros_c = []
for i in range(1, N_c+1):
 zeros_c.append(str(pari(f'Strprintf("%.50g", z_c[{i}])')))
all_zeros = all_zeros + zeros_c
print("total:", len(all_zeros), "last:", all_zeros[-1])
import numpy as np
np.save('ldelta_zeros_N3592_dps38.npy', np.array(all_zeros, dtype='<U51'))
print("saved ldelta_zeros_N3592_dps38.npy")


total: 3592 last: 2299.65084573975317969661512076437044836
saved ldelta_zeros_N3592_dps38.npy


In [27]:
# Now do the Front III analysis. Install ripser/persim
import subprocess
r = subprocess.run(['pip', 'install', '-q', 'ripser', 'persim'], capture_output=True, text=True, timeout=120)
print("RC:", r.returncode, r.stderr[-500:])


RC: 0 


In [28]:
import numpy as np
from ripser import ripser
from persim import bottleneck

# Load all imaginary-part zero lists as float64 for the analysis
def load_floats(path, field=None):
 a = np.load(path, allow_pickle=False)
 if a.dtype.names is not None and field is not None:
 s = a[field]
 else:
 s = a
 return np.array([float(x) for x in s])

zeta_im = load_floats('zeta_zeros_N5000_dps50.npy')
ldh_im = load_floats('ldh_zeros_N5000_dps50.npy')
lchi5_im = load_floats('lchi5_zeros_N5000_dps80.npy')
ldelta_im = load_floats('ldelta_zeros_N3592_dps38.npy')

# For zeta_delta_strong: structured dtype with 're','im'
zds = np.load('zeta_delta_strong_N5000_dps50.npy')
zds_re = np.array([float(x) for x in zds['re']])
zds_im = np.array([float(x) for x in zds['im']])

# For ldh off-line
import pandas as pd
ldh_off = pd.read_csv('ldh_off_line_zeros.csv')
print(ldh_off.columns.tolist())
print(ldh_off.shape)
print("zeta_im shape:", zeta_im.shape, "last:", zeta_im[-1])
print("ldh_im shape:", ldh_im.shape, "last:", ldh_im[-1])
print("lchi5_im shape:", lchi5_im.shape, "last:", lchi5_im[-1])
print("ldelta_im shape:", ldelta_im.shape, "last:", ldelta_im[-1])
print("zds_re:", zds_re[:3], "...", zds_re[1000:1002], "values at perturb idx (1000,1020,...,1380)")


['sigma', 't', 'sigma_str', 't_str', 'absL']
(110, 5)
zeta_im shape: (5000,) last: 5447.8619983012995
ldh_im shape: (5000,) last: 4981.167489862579
lchi5_im shape: (5000,) last: 4388.20901259418
ldelta_im shape: (3592,) last: 2299.650845739753
zds_re: [0.5 0.5 0.5] ... [1.5 0.5] values at perturb idx (1000,1020,...,1380)


In [29]:
# Unfolding: for each L-function, unfold gamma_i by its smooth counting function N(T).
# zeta: N(T) = (T/(2*pi)) * log(T/(2*pi*e)) + 7/8
# L(chi_4 mod 5): degree 1, conductor 5: N(T) = (T/(2*pi)) * log(5*T/(2*pi*e)) + ... (use general formula)
# General: N(T) ~ (T/(2*pi)) log(q*T^d / (2*pi*e)^d), where d=degree, q=conductor
# L_DH: degree 1, conductor 5 (Dirichlet-like; not L-function but same Hurwitz combination). Same as L(chi_4 mod 5).
# Actually L_DH = ? Davenport-Heilbronn is a combination of two Hurwitz zetas with character mod 5, so density same as L(chi mod 5).
# L(Delta,s): degree 2, conductor 1, weight 12 (analytic norm: shift to Re(s)=1/2): N(T) ~ (T/pi) * log(T/(2*pi*e))

# Let me unfold each list using the smooth main term
def unfold_zeta(t):
 return (t/(2*np.pi)) * np.log(t/(2*np.pi*np.e)) + 7/8

def unfold_deg1(t, q):
 return (t/(2*np.pi)) * np.log(q*t/(2*np.pi*np.e))

def unfold_deg2_level1(t):
 # degree 2, q=1: N(T) ~ 2 * (T/(2*pi)) log(T/(2*pi*e)) = (T/pi)*log(T/(2*pi*e))
 return (t/np.pi) * np.log(t/(2*np.pi*np.e))

uf_zeta = unfold_zeta(zeta_im)
uf_ldh = unfold_deg1(ldh_im, q=5)
uf_lchi5 = unfold_deg1(lchi5_im, q=5)
uf_ldelta = unfold_deg2_level1(ldelta_im)

# Spacings
def spacings(uf):
 s = np.diff(uf)
 s = s[s>0] # safety
 return s

s_zeta = spacings(uf_zeta)
s_ldh = spacings(uf_ldh)
s_lchi5 = spacings(uf_lchi5)
s_ldelta = spacings(uf_ldelta)

for n,s in [('zeta',s_zeta),('ldh',s_ldh),('lchi5',s_lchi5),('ldelta',s_ldelta)]:
 print(f"{n}: N={len(s)}, mean={s.mean():.4f}, std={s.std():.4f}, min={s.min():.4f}, max={s.max():.4f}")


zeta: N=4999, mean=1.0000, std=0.3896, min=0.0463, max=2.4679
ldh: N=4999, mean=1.1552, std=0.5942, min=0.0827, max=5.7304
lchi5: N=4999, mean=0.9999, std=0.3725, min=0.0996, max=2.5315
ldelta: N=3591, mean=0.9999, std=0.3665, min=0.1230, max=2.2209


In [30]:
# Note L_DH unfolding mean is 1.155 instead of 1.000 — suggests our deg-1 q=5 model isn't right.
# Davenport-Heilbronn density: actually L_DH has functional equation with conductor q=5 but the density formula needs checking.
# Let's compute the empirical local mean to rescale, ensuring all are mean-1.
# For Front III the absolute scale doesn't matter as long as we compare normalized spacings.

# Renormalize so mean=1
s_zeta = s_zeta / s_zeta.mean()
s_ldh = s_ldh / s_ldh.mean()
s_lchi5 = s_lchi5 / s_lchi5.mean()
s_ldelta = s_ldelta / s_ldelta.mean()
print({n: float(s.mean()) for n,s in [('zeta',s_zeta),('ldh',s_ldh),('lchi5',s_lchi5),('ldelta',s_ldelta)]})

# Sliding window embedding (3D point cloud), as in f10 protocol
def sliding_window(x, dim=3, stride=1):
 N = len(x) - (dim-1)*stride
 return np.stack([x[i*stride:i*stride+N] for i in range(dim)], axis=1)

# Subsample for tractable ripser computation (full 5000 spacings -> ~5000 points in 3D is heavy)
# Use 1500 points per function for permutation test
def make_cloud(s, n=1500, seed=0):
 rng = np.random.default_rng(seed)
 cloud = sliding_window(s, 3)
 if len(cloud) > n:
 idx = rng.choice(len(cloud), n, replace=False)
 cloud = cloud[np.sort(idx)]
 return cloud

# Test ripser on one
import time
cloud = make_cloud(s_zeta, 800)
print("cloud shape:", cloud.shape)
t0 = time.time()
res = ripser(cloud, maxdim=1)
print("time:", time.time()-t0, "H1 features:", len(res['dgms'][1]))


{'zeta': 1.0, 'ldh': 1.0, 'lchi5': 0.9999999999999998, 'ldelta': 1.0}
cloud shape: (800, 3)


time: 0.5870709419250488 H1 features: 398


In [31]:
# Good - 0.59s per ripser run on 800 points. For permutation test B=1000, need 2000 ripsers per pair.
# 6 pairs * 2000 = 12000 ripsers * 0.59s = 7080s — too slow.
# Reduce to n=500 and B=300. Also bottleneck can be costly. Let me time bottleneck.
cloud2 = make_cloud(s_lchi5, 800, seed=1)
res2 = ripser(cloud2, maxdim=1)
t0=time.time()
bd = bottleneck(res['dgms'][1], res2['dgms'][1])
print("bottleneck:", bd, "t:", time.time()-t0)


bottleneck: 0.04982385039329529 t: 3.8079357147216797


In [32]:
# Bottleneck is 3.8s per call. That's much slower. Need to reduce.
# Use smaller clouds and Wasserstein or persistence summaries. Alternative: use a statistic on H1 directly.
# 
# Strategy: instead of permutation test using bottleneck distance, use a permutation test
# on a summary statistic of H1 diagram (e.g. total persistence) which is fast to compute.
# This still tests "are the H1 distributions the same".
# 
# Better: use a permutation test on a simple summary derived from the cloud + H1.
# Plan: 
# 1) Compute observed bottleneck distance between pairs.
# 2) For each pair, do a label-permutation test: pool samples, randomly relabel B times,
# compute summary statistic = total persistence of larger H1.
# This is computationally heavy; use n=400, B=200.

# Even faster: use Wasserstein-2 on diagrams? Or just compare persistence entropy as scalar.
# 
# Practical approach given remaining time (~600s):
# - Compute H1 persistence diagrams for each of the 4 functions on n=600 sliding window points (~0.4s each)
# - For each pair, permutation test using statistic = |TotalPersistence(H1_A) - TotalPersistence(H1_B)|
# * Permutation: pool spacings, split randomly, recompute H1 total persistence each side, take diff.
# 
# Even this is 6 pairs * B=200 * 2 ripsers = 2400 ripsers @ 0.4s = 960s. Too slow.
# 
# Reduce to n=400, B=100, that's 6*100*2 = 1200 ripsers @ 0.2s ~ 240s. Manageable.

def total_persistence(dgm):
 if len(dgm) == 0:
 return 0.0
 # finite features only
 finite = dgm[np.isfinite(dgm[:,1])]
 if len(finite) == 0:
 return 0.0
 return float(np.sum(finite[:,1] - finite[:,0]))

def stat_from_spacings(s, n=400, seed=0, dim=3):
 rng = np.random.default_rng(seed)
 cloud = sliding_window(s, dim)
 if len(cloud) > n:
 idx = rng.choice(len(cloud), n, replace=False)
 cloud = cloud[np.sort(idx)]
 res = ripser(cloud, maxdim=1)
 return total_persistence(res['dgms'][1]), res['dgms'][1]

# Compute observed for all 4
results = {}
import time
t0 = time.time()
for name, s in [('zeta', s_zeta), ('lchi5', s_lchi5), ('ldh', s_ldh), ('ldelta', s_ldelta)]:
 tp, dgm = stat_from_spacings(s, n=600, seed=42)
 results[name] = (tp, dgm)
 print(f"{name}: TP_H1={tp:.4f}, #H1 features={len(dgm)}")
print("time:", time.time()-t0)


zeta: TP_H1=9.4044, #H1 features=290


lchi5: TP_H1=7.9755, #H1 features=288


ldh: TP_H1=6.1287, #H1 features=239


ldelta: TP_H1=7.6873, #H1 features=275
time: 1.0031614303588867


In [33]:
# Permutation test: for pair (A, B), pool samples and reshuffle B times.
# Statistic: |TP(H1_A) - TP(H1_B)|

import time
names = ['zeta', 'lchi5', 'ldh', 'ldelta']
samples = {'zeta': s_zeta, 'lchi5': s_lchi5, 'ldh': s_ldh, 'ldelta': s_ldelta}

# Use a consistent N for each sample by truncating
N_use = min(len(samples[n]) for n in names)
print("N_use:", N_use)
for n in names:
 samples[n] = samples[n][:N_use]

# n_cloud per side, B perms
n_cloud = 500
B = 200

def tp_from_array(arr, n=n_cloud, seed=0):
 rng = np.random.default_rng(seed)
 cloud = sliding_window(arr, 3)
 if len(cloud) > n:
 idx = rng.choice(len(cloud), n, replace=False)
 cloud = cloud[np.sort(idx)]
 res = ripser(cloud, maxdim=1)
 return total_persistence(res['dgms'][1])

# Observed stat for each pair using fixed seed
def observed_stat(A, B):
 tpA = tp_from_array(A, seed=0)
 tpB = tp_from_array(B, seed=0)
 return abs(tpA - tpB), tpA, tpB

import itertools
pairs = list(itertools.combinations(names, 2))
print("Pairs:", pairs)

# Estimate timing: 6 pairs * 200 perms * 2 ripsers = 2400 ripsers
# At n=500 ripser is ~0.3s; ~720s. Too long.
# Reduce B to 100 and n to 400: 6*100*2 = 1200 * 0.2 = 240s.
n_cloud = 400
B = 100

t0=time.time()
tp = tp_from_array(s_zeta, n=n_cloud, seed=0)
print("single ripser at n=400:", time.time()-t0, "tp:", tp)


N_use: 3591
Pairs: [('zeta', 'lchi5'), ('zeta', 'ldh'), ('zeta', 'ldelta'), ('lchi5', 'ldh'), ('lchi5', 'ldelta'), ('ldh', 'ldelta')]
single ripser at n=400: 0.12587308883666992 tp: 6.263708420097828


In [34]:
# 0.126s per ripser - excellent. 6 pairs * 100 perms * 2 ripsers = 1200 * 0.126 = 151s. Feasible.
# Boost to B=200 -> 302s.

n_cloud = 400
B = 200

# Pre-cache observed stats
observed = {}
for a, b in pairs:
 tpa = tp_from_array(samples[a], n=n_cloud, seed=0)
 tpb = tp_from_array(samples[b], n=n_cloud, seed=1)
 observed[(a,b)] = (abs(tpa-tpb), tpa, tpb)
 print(f"obs {a} vs {b}: |dTP|={abs(tpa-tpb):.4f}, TP_a={tpa:.3f}, TP_b={tpb:.3f}")


obs zeta vs lchi5: |dTP|=0.5120, TP_a=5.994, TP_b=5.482


obs zeta vs ldh: |dTP|=2.7462, TP_a=5.994, TP_b=3.247


obs zeta vs ldelta: |dTP|=0.0033, TP_a=5.994, TP_b=5.990


obs lchi5 vs ldh: |dTP|=2.3352, TP_a=5.583, TP_b=3.247


obs lchi5 vs ldelta: |dTP|=0.4078, TP_a=5.583, TP_b=5.990


obs ldh vs ldelta: |dTP|=2.5401, TP_a=3.450, TP_b=5.990


In [35]:
# Clear pattern! L_DH has much lower TP(H1) than the others - it's distinguishable.
# zeta, lchi5, ldelta all similar (5-6). ldh is ~3.3.
# 
# Now the permutation test. For each pair (A,B):
# - Pool 2*N_use spacings
# - For each permutation: random split into two halves of equal size
# - Compute |TP(H1_A_perm) - TP(H1_B_perm)|
# - p = (1 + #|stat_perm| >= |stat_obs|) / (1+B)

import time
rng_master = np.random.default_rng(2024)
p_values = {}
overall_t0 = time.time()

for a, b in pairs:
 A = samples[a]
 B_arr = samples[b]
 pooled = np.concatenate([A, B_arr])
 n = len(A)
 obs_stat = observed[(a,b)][0]
 
 perm_stats = np.zeros(B)
 t0 = time.time()
 for k in range(B):
 rng_master.shuffle(pooled)
 A_p = pooled[:n].copy()
 B_p = pooled[n:].copy()
 tpa = tp_from_array(A_p, n=n_cloud, seed=k+100)
 tpb = tp_from_array(B_p, n=n_cloud, seed=k+1000)
 perm_stats[k] = abs(tpa - tpb)
 p = (1 + np.sum(perm_stats >= obs_stat)) / (1 + B)
 p_values[(a,b)] = (p, obs_stat, perm_stats.mean(), perm_stats.std())
 print(f"{a} vs {b}: obs={obs_stat:.4f}, perm_mean={perm_stats.mean():.4f}, perm_std={perm_stats.std():.4f}, p={p:.4f}, t={time.time()-t0:.1f}s")
print("total:", time.time()-overall_t0, "s")


TimeoutError: Code execution timed out after 32.0 seconds